# UC Xenium preprocessing

Turns the raw Single-Cell-Portal exports in `UC_Xenium_Data_Raw/` into one
`adata.h5ad` per sample, laid out to mirror the existing `Xenium/` dataset folder:

```
UC_Xenium/
  Sample_1_UC1_inflamed/adata.h5ad        # inflamed
  Sample_2_UC1_less_inflamed/adata.h5ad   # less inflamed
```

Each adata holds **raw counts in `.X`** (no normalized layer — SpaceTravLR and
harreman build their own), `.obs['cell_type']`, `.obsm['spatial']`, and the
dataset's precomputed `.obsm['X_umap']` (so SpaceTravLR's `setup_` skips
recomputing PCA/neighbors/UMAP).


In [ ]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import os
import gzip

import numpy as np
import pandas as pd
import anndata as ad

from metab_processing.metab_travlr_config import DATA_DIR

In [ ]:
# Raw exports live next to the Xenium folder; adata objects go in UC_Xenium/.
RAW_DIR = f'{DATA_DIR}/UC_Xenium_Data_Raw'
OUT_DIR = f'{DATA_DIR}/UC_Xenium'

# folder name -> (biosample_id in metadata, its spatial-coordinate file)
SAMPLES = {
    'Sample_1_UC1_inflamed':      ('UC1_inflamed',      'spatial_UC1_1.tsv'),
    'Sample_2_UC1_less_inflamed': ('UC1_less_inflamed', 'spatial_UC1_2.tsv'),
}

In [ ]:
# Shared inputs (all cells from both samples).
# The .tsv files have a second 'TYPE' header row we skip; NAME is the global cell index.
meta = pd.read_csv(f'{RAW_DIR}/spatial_metadata.tsv', sep='\t', skiprows=[1], index_col=0)

# Precomputed UMAP provided with the dataset (all cells); saved so SpaceTravLR
# skips recomputing PCA/neighbors/UMAP on these large samples.
umap = pd.read_csv(f'{RAW_DIR}/spatial_cluster_umap.tsv', sep='\t', skiprows=[1], index_col=0)

# Dense counts are genes x cells and gzip-compressed despite the .tsv name. With ~871k
# COLUMNS this file is pathological for pandas (even reading the header costs many minutes),
# so stream it with numpy instead: the header is just cell indices 0..N-1 (= global cell
# index, in order) and each of the ~1.5k gene rows parses straight to an int16 vector
# (Xenium counts are tiny). ~1 minute total instead of pandas' many minutes.
counts_path = f'{RAW_DIR}/spatial_raw_counts_dense.tsv'
genes, rows = [], []
with gzip.open(counts_path, 'rt') as f:
    header = f.readline().rstrip('\n').split('\t')          # 'GENE' + cell indices
    for line in f:
        i = line.index('\t')                                # first tab ends the gene name
        genes.append(line[:i])
        rows.append(np.array(line[i + 1:].rstrip('\n').split('\t'), dtype=np.int16))
counts = np.vstack(rows)                                     # genes x cells, int16

cell_ids = header[1:]
assert cell_ids == [str(i) for i in range(len(cell_ids))], \
    'expected counts columns to be cell indices 0..N-1 in order'
assert counts.shape == (len(genes), len(meta))
print('metadata:', meta.shape, '| counts (genes x cells):', counts.shape)

In [ ]:
# Cell columns are global cell index 0..N-1 in order, so a sample's metadata index values
# double as its column positions in `counts`.
var = pd.DataFrame(index=genes)

for folder, (biosample, coord_file) in SAMPLES.items():
    cell_idx = meta.index[meta['biosample_id'] == biosample].to_numpy()
    coords = pd.read_csv(f'{RAW_DIR}/{coord_file}', sep='\t', skiprows=[1], index_col=0)

    obs = meta.loc[cell_idx].copy()
    obs['cell_type'] = obs['cell_types']

    adata = ad.AnnData(
        X=counts[:, cell_idx].T.astype('float32'),   # cells x genes, raw counts
        obs=obs,
        var=var.copy(),
    )
    adata.obs_names = cell_idx.astype(str)
    adata.var_names_make_unique()
    adata.obsm['spatial'] = coords.loc[cell_idx, ['X', 'Y']].to_numpy()
    adata.obsm['X_umap'] = umap.loc[cell_idx, ['X', 'Y']].to_numpy()

    out = f'{OUT_DIR}/{folder}'
    os.makedirs(out, exist_ok=True)
    adata.write_h5ad(f'{out}/adata.h5ad')
    print(folder, '->', adata.shape)
    print(adata)